# UniChart — Bar Overlay Styles: `marker`, `tick`, `whisker`

Bar plots can overlay extra columns on top of the bars via `nb.bar(..., markers=...)`.
Historically each overlay column rendered as a **marker symbol**. This demo shows the
new **`style`** key of `var_format`, which lets an overlay column render as:

| style | glyph |
|---|---|
| `'marker'` *(default)* | classic marker symbol at the value |
| `'tick'` | horizontal dash at the value (bullet-chart target) |
| `'whisker'` | dash **plus a stem** connecting it to the top of its bar |

For whiskers, `linestyle` sets the stem's dash pattern (`'-'`, `'--'`, `'-.'`, `':'`;
`'None'` hides the stem) and `linewidth` its thickness.

Covered:

- **A. Baseline** — the classic marker overlay, unchanged.
- **B. `style='tick'`** — a dash at the value, per bar.
- **C. `style='whisker'`** — dash + stem to the bar top; stems point **down or up**
  depending on whether the value sits above or below its bar.
- **D. Stem styling** — `linestyle` / `linewidth` control the whisker stem.
- **E. Mixed styles** — one column as a whisker, another as a tick, on the same bars.
- **F. `by='sets'`** — whiskers in the per-dataset subplot view.
- **G. API details** — validation, `style='reset'`, persistence.

**Interactive:** each section leaves its `UnichartNotebook` in a top-level variable
(`uc_base`, `uc_tick`, `uc_whisker`, `uc_stem`, `uc_mixed`, `uc_sets`) so you can
keep experimenting after running all cells.

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import pandas as pd
from unichart import UnichartNotebook

PASSED = 0
def check(cond, msg):
    global PASSED
    assert cond, 'FAILED: ' + msg
    PASSED += 1
    print('PASS:', msg)

# Per-phase engine data: EGT bars, a red-line EGT_LIMIT above the bars, and an
# EGT_TARGET that is deliberately ABOVE some bar tops and BELOW others, so the
# whisker's two stem directions are both visible.
PHASES = ['TAXI', 'TAKEOFF', 'CLIMB', 'CRUISE', 'DESCENT']
ENG_A = pd.DataFrame({
    'PHASE': PHASES,
    'EGT':        [420, 880, 810, 700, 520],
    'EGT_LIMIT':  [500, 950, 900, 800, 600],
    'EGT_TARGET': [430, 860, 790, 720, 500],
})
ENG_B = pd.DataFrame({
    'PHASE': PHASES,
    'EGT':        [450, 910, 840, 730, 540],
    'EGT_LIMIT':  [500, 950, 900, 800, 600],
    'EGT_TARGET': [430, 860, 790, 720, 500],
})

def build():
    nb = UnichartNotebook()
    nb.load_df(ENG_A, title='Engine A')
    nb.load_df(ENG_B, title='Engine B')
    return nb

def caps_of(fig, name):
    """The overlay column's cap/marker traces (excludes companion stem traces)."""
    return [t for t in fig.data if t.type == 'scatter'
            and name in (t.name or '') and '(stem)' not in (t.name or '')]

def stems_of(fig, name):
    """Companion line traces drawn for dashed whisker stems."""
    return [t for t in fig.data if t.type == 'scatter'
            and name in (t.name or '') and '(stem)' in (t.name or '')]

## A. Baseline — classic marker overlay (unchanged)

Without a `style` override, `markers=` behaves exactly as before: a marker symbol
at each value, one legend entry per dataset.

In [2]:
uc_base = build()
fig = uc_base.bar(x='PHASE', y='EGT', markers='EGT_LIMIT',
                  suptitle='A. Baseline marker overlay')

tr = caps_of(fig, 'EGT_LIMIT')
check(len(tr) == 2, 'one marker trace per dataset')
check(all(t.marker.symbol not in ('line-ew',) for t in tr), 'baseline uses a symbol, not a dash')
check(all(t.error_y.array is None for t in tr), 'baseline has no whisker stem')
fig.show()

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
PASS: one marker trace per dataset
PASS: baseline uses a symbol, not a dash
PASS: baseline has no whisker stem


## B. `style='tick'` — a dash at the value

`nb.var_format(col, style='tick')` swaps the symbol for a horizontal dash that sits
over its own bar (bullet-chart "target" look). All the other `var_format` keys still
apply — here the column is also pinned red, which gives it a single shared legend
entry (the usual behavior for explicitly styled overlay columns).

In [3]:
uc_tick = build()
uc_tick.var_format('EGT_LIMIT', style='tick', color='red')
fig = uc_tick.bar(x='PHASE', y='EGT', markers='EGT_LIMIT',
                  suptitle="B. style='tick'")

tr = caps_of(fig, 'EGT_LIMIT')
check(all(t.marker.symbol == 'line-ew' for t in tr), 'tick renders as a horizontal dash')
check(all(t.error_y.array is None for t in tr), 'tick has no stem')
fig.show()

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
PASS: tick renders as a horizontal dash
PASS: tick has no stem


## C. `style='whisker'` — dash + stem to the bar top

The whisker adds a vertical stem from the dash down (or up) to the top of the bar it
belongs to, so the *gap* between the bar and the value is what your eye reads.
Alignment with grouped bars is automatic: the overlay inherits the bar's
`offsetgroup`, and the layout enables `scattermode='group'` when `barmode='group'`.

In [4]:
uc_whisker = build()
uc_whisker.var_format('EGT_LIMIT', style='whisker', color='red')
fig = uc_whisker.bar(x='PHASE', y='EGT', markers='EGT_LIMIT',
                     suptitle="C. style='whisker'")

tr = caps_of(fig, 'EGT_LIMIT')
check(all(t.error_y.array is not None for t in tr), 'whisker carries stem lengths (error_y)')
# LIMIT is above every bar, so every stem points DOWN (arrayminus) and none up.
check(all(max(t.error_y.array) == 0 for t in tr), 'no upward stems when value is above the bar')
check(all(min(t.error_y.arrayminus) > 0 for t in tr), 'downward stems span limit -> bar top')
check(fig.layout.scattermode == 'group', 'overlays align over their own grouped bar')
fig.show()

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
PASS: whisker carries stem lengths (error_y)
PASS: no upward stems when value is above the bar
PASS: downward stems span limit -> bar top
PASS: overlays align over their own grouped bar


## D. Stem styling — `linestyle` / `linewidth`

The whisker stem takes its dash pattern from `linestyle` and its thickness from
`linewidth`. Solid stems ride on the cap trace's error bars; **dashed** stems are
drawn as a companion line trace (Plotly error bars can only be solid), tied to the
same legend group so legend clicks toggle cap and stem together.
`linestyle='None'` suppresses the stem entirely.

In [5]:
uc_stem = build()
uc_stem.var_format('EGT_LIMIT', style='whisker', color='red', linestyle='--', linewidth=3)
fig = uc_stem.bar(x='PHASE', y='EGT', markers='EGT_LIMIT',
                  suptitle="D. whisker, linestyle='--', linewidth=3")

caps, stems = caps_of(fig, 'EGT_LIMIT'), stems_of(fig, 'EGT_LIMIT')
check(len(stems) == 2, 'dashed stems draw as companion line traces (one per dataset)')
check(all(t.line.dash == 'dash' and t.line.width == 3 for t in stems),
      'stem takes its dash and width from linestyle/linewidth')
check(all(t.error_y.array is None for t in caps), 'no error-bar stem when the stem is dashed')

# linewidth alone thickens the (solid) error-bar stem
nb2 = build()
nb2.var_format('EGT_LIMIT', style='whisker', linewidth=5)
f2 = nb2.bar(x='PHASE', y='EGT', markers='EGT_LIMIT')
check(all(t.error_y.thickness == 5 for t in caps_of(f2, 'EGT_LIMIT')),
      'solid stems honor linewidth via error-bar thickness')

# linestyle='None' hides the stem entirely (bare cap)
nb3 = build()
nb3.var_format('EGT_LIMIT', style='whisker', linestyle='None')
f3 = nb3.bar(x='PHASE', y='EGT', markers='EGT_LIMIT')
check(all(t.error_y.array is None for t in caps_of(f3, 'EGT_LIMIT'))
      and not stems_of(f3, 'EGT_LIMIT'),
      "linestyle='None' hides the stem entirely")
fig.show()

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
PASS: dashed stems draw as companion line traces (one per dataset)
PASS: stem takes its dash and width from linestyle/linewidth
PASS: no error-bar stem when the stem is dashed
UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
PASS: solid stems honor linewidth via error-bar thickness
UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
PASS: linestyle='None' hides the stem entirely


## E. Mixed styles on the same bars

Each overlay column chooses its own style: the red **limit** renders as a whisker
(dashed stem), the green **target** as a tick. In CRUISE the target sits *below*
the bar tops — note the tick simply sits inside the bar, still legible.

In [6]:
uc_mixed = build()
uc_mixed.var_format('EGT_LIMIT',  style='whisker', color='red', linestyle=':')
uc_mixed.var_format('EGT_TARGET', style='tick',    color='green')
fig = uc_mixed.bar(x='PHASE', y='EGT', markers=['EGT_LIMIT', 'EGT_TARGET'],
                   suptitle='E. whisker (dotted stem) + tick together')

check(len(caps_of(fig, 'EGT_LIMIT')) == 2 and len(caps_of(fig, 'EGT_TARGET')) == 2,
      'both overlay columns drawn for both datasets')
check(all(t.line.dash == 'dot' for t in stems_of(fig, 'EGT_LIMIT')),
      'the limit stem is dotted while the target stays a tick')
fig.show()

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
PASS: both overlay columns drawn for both datasets
PASS: the limit stem is dotted while the target stays a tick


## F. `by='sets'` — whiskers in the per-dataset view

In the per-dataset view color encodes *variable*, so an overlay column is not tied
to a single bar series; tick/whisker glyphs attach to the **first** plotted y
variable's bars (classic markers stay at the category center, as before).
Stem styling works here too.

In [7]:
uc_sets = build()
uc_sets.var_format('EGT_LIMIT', style='whisker', color='red', linestyle='--')
fig = uc_sets.bar(x='PHASE', y='EGT', markers='EGT_LIMIT', by='sets',
                  suptitle="F. by='sets', whisker with dashed stem")

check(len(caps_of(fig, 'EGT_LIMIT')) == 2 and len(stems_of(fig, 'EGT_LIMIT')) == 2,
      'per-dataset view draws a cap and a dashed stem per subplot')
fig.show()

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
PASS: per-dataset view draws a cap and a dashed stem per subplot


## G. API details — validation, reset, persistence

`style` is validated up front, persists like any other `var_format` attribute, and
`style='reset'` clears just that key (falling back to the classic marker).

In [8]:
nb = build()

try:
    nb.var_format('EGT_LIMIT', style='sparkles')
    check(False, 'invalid style should raise')
except ValueError as e:
    check('style must be one of' in str(e), 'invalid style raises ValueError')

nb.var_format('EGT_LIMIT', style='whisker')
check(nb.variable_formats['EGT_LIMIT'].get('style') == 'whisker', 'style persists in variable_formats')

nb.var_format('EGT_LIMIT', style='reset')
check('style' not in nb.variable_formats.get('EGT_LIMIT', {}), "style='reset' clears the override")

fig = nb.bar(x='PHASE', y='EGT', markers='EGT_LIMIT', suptitle='G. after reset: classic marker again')
check(all(t.error_y.array is None for t in caps_of(fig, 'EGT_LIMIT')),
      'after reset the overlay is a plain marker again')

print(f'\nAll {PASSED} checks passed.')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
PASS: invalid style raises ValueError
PASS: style persists in variable_formats
PASS: style='reset' clears the override
PASS: after reset the overlay is a plain marker again

All 21 checks passed.
